# 5.1 · Client features: elicitation, roots and sampling

**客户端特性：征询、根目录与采样**

> **After this lesson you can:** Explain what elicitation, roots and sampling let a server ask of the client, how each now travels inside an InputRequiredResult, and migrate deprecated roots and sampling to tool parameters and direct LLM calls.

Official reading: [Understanding MCP clients](https://modelcontextprotocol.io/docs/2026-07-28/learn/client-concepts) · [Spec: Roots](https://modelcontextprotocol.io/specification/2026-07-28/client/roots) · [Spec: Sampling](https://modelcontextprotocol.io/specification/2026-07-28/client/sampling)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. Setup: what a client declares depends on its callbacks
In the Python SDK, you declare a client feature by **passing a callback**. `Client(elicitation_callback=...)` advertises `elicitation: {form, url}`, `list_roots_callback=` advertises `roots`, and `sampling_callback=` advertises `sampling`. The server sees these in the request's `_meta.io.modelcontextprotocol/clientCapabilities`. Let's prove it with a tool that echoes what it received.

In [ ]:
import json
import logging
import warnings

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.getLogger("mcp").setLevel(logging.CRITICAL)    # keep the SDK's own server-side logs quiet
warnings.filterwarnings("ignore", message=".*deprecated.*")


def wire(model) -> str:
    """A pydantic MCP object exactly as it travels on the wire."""
    return json.dumps(model.model_dump(by_alias=True, exclude_none=True, mode="json"), indent=2)


probe = MCPServer("capability-probe")


@probe.tool()
def my_capabilities(ctx: Context) -> str:
    """Return the client capabilities declared on this request."""
    caps = ctx.client_capabilities
    return json.dumps(caps.model_dump(by_alias=True, exclude_none=True) if caps else {})


async def noop(*args):          # placeholder callbacks: we only care about what gets declared
    return types.ErrorData(code=-1, message="unused")

async with Client(probe) as plain:
    print("no callbacks      ->", (await plain.call_tool("my_capabilities", {})).content[0].text)

async with Client(probe, elicitation_callback=noop, list_roots_callback=noop, sampling_callback=noop) as rich:
    print("all three         ->", (await rich.call_tool("my_capabilities", {})).content[0].text)

### 2. Elicitation: a travel server asks for booking details
This is the docs' Barcelona example. In `MCPServer`, the idiomatic 2026 way to ask a question is a **resolver**: a function that returns `Elicit(message, Schema)`. You attach it to a tool parameter with `Annotated[..., Resolve(fn)]`. The framework turns the question into an `InputRequiredResult`, and when the client retries with the answer it injects the result and runs the tool body. Annotating the parameter as `ElicitationResult[Schema]` gives you all three outcomes to branch on.

The Pydantic model is rendered as a **restricted flat schema**: a boolean, a string enum (`Literal`) and a boolean with a default. A nested model would be rejected.

In [ ]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field

from mcp.server.elicitation import AcceptedElicitation, ElicitationResult
from mcp.server.mcpserver import Elicit, Resolve


class BookingDetails(BaseModel):
    confirmBooking: bool = Field(description="Confirm the booking (Flights + Hotel = $3,000)")
    seatPreference: Literal["window", "aisle", "no preference"] = Field(description="Preferred seat type")
    travelInsurance: bool = Field(default=False, description="Add travel insurance ($150)")


travel = MCPServer("travel")


def ask_booking_details(destination: str) -> Elicit[BookingDetails]:
    return Elicit(f"Please confirm your {destination} vacation booking details:", BookingDetails)


@travel.tool()
def book_trip(destination: str,
              details: Annotated[ElicitationResult[BookingDetails], Resolve(ask_booking_details)]) -> str:
    """Book a vacation package after the user confirms the details."""
    if isinstance(details, AcceptedElicitation):
        d = details.data
        if not d.confirmBooking:
            return "User reviewed the form but did not confirm. Nothing booked."
        extra = " + insurance" if d.travelInsurance else ""
        return f"Booked {destination}: {d.seatPreference} seat{extra}."
    return f"Booking not made: user chose '{details.action}'."


async with Client(travel) as c:
    tool = (await c.list_tools()).tools[0]
    print("book_trip input schema (what the LLM fills):", json.dumps(tool.input_schema))

Now the **client side**. A real host would render a form. Our scripted "user" prints what a UI would show (which server is asking, the message, the fields) and then answers. We run it three times to see `accept`, `decline` and `cancel`.

In [ ]:
script = []          # the scripted user's next answers


async def form_ui(context, params: types.ElicitRequestFormParams):
    """Stand-in for a host's elicitation dialog."""
    print(f"  ┌ [travel server] asks: {params.message}")
    for name, prop in params.requested_schema["properties"].items():
        choices = prop.get("enum")
        print(f"  │  {name:<16} {prop['type']:<8} {choices or ''} default={prop.get('default', '-')}")
    action, content = script.pop(0)
    print(f"  └ user -> {action} {content or ''}")
    return types.ElicitResult(action=action, content=content)


script[:] = [("accept", {"confirmBooking": True, "seatPreference": "window", "travelInsurance": True}),
             ("decline", None),
             ("cancel", None)]

async with Client(travel, elicitation_callback=form_ui) as client:
    for _ in range(3):
        result = await client.call_tool("book_trip", {"destination": "Barcelona"})
        print("tool result:", result.content[0].text, "\n")

### 3. Look under the hood: the input request on the wire
`client.call_tool` hid the round trip. `client.session.call_tool(..., allow_input_required=True)` returns the raw interim result instead, which here is the `InputRequiredResult` with the `elicitation/create` request *inside* it. Lesson 5.3 drives this loop by hand. For now, look at the shape.

In [ ]:
async with Client(travel, elicitation_callback=form_ui) as client:
    interim = await client.session.call_tool("book_trip", {"destination": "Barcelona"}, allow_input_required=True)
    print(type(interim).__name__)
    text = wire(interim)
    print(text[:1400] + ("\n  ..." if len(text) > 1400 else ""))

Notice `resultType: "input_required"`, the server-chosen key in `inputRequests`, and a `requestState` the SDK has **encrypted**. The client must echo it back untouched.

### 4. Break it: no capability, no question
A server **MUST NOT** send an input request the client didn't declare. Connect *without* an `elicitation_callback` and the SDK refuses with the spec's `-32021` MissingRequiredClientCapability error, naming what is missing.

In [ ]:
from mcp.shared.exceptions import MCPError

async with Client(travel) as client:          # no elicitation_callback -> no elicitation capability
    try:
        await client.call_tool("book_trip", {"destination": "Barcelona"})
    except MCPError as err:
        print("code:", err.error.code)
        print("message:", err.error.message)
        print("data:", err.error.data)

### 5. Sampling (deprecated): the server borrows the host's model
Sampling still works during the deprecation window. A resolver returns `Sample(...)` with messages, a required `max_tokens` and `model_preferences`. The client's `sampling_callback` is where a host would show the prompt for approval, **pick a model** from the hints and priorities, call its LLM and show the answer for review. Our fake host has three models and does the hint matching the spec describes (substring, in order, then priorities).

In [ ]:
from mcp.server.mcpserver import Sample

flights = MCPServer("flights")
FLIGHTS = "IB3101 07:10 €180 direct | VY1307 06:30 €95 1 stop | BA478 21:40 €70 red-eye"


def analyse(preferences: str) -> Sample:
    return Sample(
        [types.SamplingMessage(role="user", content=types.TextContent(
            type="text", text=f"Recommend the best flight.\n{FLIGHTS}\nUser preferences: {preferences}"))],
        max_tokens=200,
        system_prompt="You are a travel expert.",
        model_preferences=types.ModelPreferences(
            hints=[types.ModelHint(name="sonnet"), types.ModelHint(name="claude")],
            cost_priority=0.3, speed_priority=0.2, intelligence_priority=0.9),
    )


@flights.tool()
def find_best_flight(preferences: str,
                     advice: Annotated[types.CreateMessageResult, Resolve(analyse)]) -> str:
    """Recommend a flight using the client's LLM (deprecated sampling)."""
    return f"[{advice.model}] {advice.content.text}"


HOST_MODELS = {"claude-opus-5-5": {"intelligence": 0.95, "speed": 0.4, "cost": 0.2},
               "claude-haiku-fast": {"intelligence": 0.6, "speed": 0.95, "cost": 0.9},
               "gemini-pro": {"intelligence": 0.9, "speed": 0.6, "cost": 0.5}}


def choose_model(prefs: types.ModelPreferences | None) -> str:
    for hint in (prefs.hints or []) if prefs else []:
        matches = [m for m in HOST_MODELS if hint.name and hint.name in m]
        if matches:
            return matches[0]
    p = prefs or types.ModelPreferences()
    score = lambda m: ((p.intelligence_priority or 0) * HOST_MODELS[m]["intelligence"] +
                       (p.speed_priority or 0) * HOST_MODELS[m]["speed"] +
                       (p.cost_priority or 0) * HOST_MODELS[m]["cost"])
    return max(HOST_MODELS, key=score)


async def host_llm(context, params: types.CreateMessageRequestParams):
    model = choose_model(params.model_preferences)
    print(f"  [host] approve sampling? prompt={params.messages[0].content.text.splitlines()[0]!r} "
          f"maxTokens={params.max_tokens} -> approved, using {model}")
    reply = "VY1307: morning departure, one stop, cheapest option that fits."   # a deterministic 'LLM'
    return types.CreateMessageResult(role="assistant", model=model, stop_reason="endTurn",
                                     content=types.TextContent(type="text", text=reply[: params.max_tokens * 4]))


async with Client(flights, sampling_callback=host_llm) as client:
    interim = await client.session.call_tool("find_best_flight", {"preferences": "morning, max 1 stop"},
                                              allow_input_required=True)
    request = next(iter(interim.input_requests.values()))
    print("input request on the wire:\n", wire(request)[:900])
    final = await client.call_tool("find_best_flight", {"preferences": "morning, max 1 stop"})
    print("\nfinal:", final.content[0].text)

There are no "sonnet" models in our host, so the first hint misses and `"claude"` matches. That is exactly the advisory, substring-based behaviour the spec describes.

### 6. Same server, old protocol: legacy mode uses server-initiated requests
With `mode="legacy"`, the client speaks 2025-11-25 and the SDK sends `sampling/createMessage` as a real **server-to-client request** in the middle of the call, which is the pre-MRTR way. The resolver code doesn't change. The old imperative helpers like `ctx.elicit(...)` only work in that legacy era, though. On a 2026-07-28 connection there is no back-channel, and they fail.

In [ ]:
legacy_demo = MCPServer("legacy-demo")


class Answer(BaseModel):
    name: str


@legacy_demo.tool()
async def ask_name_old_style(ctx: Context) -> str:
    """Ask for a name the pre-2026 way (server-initiated elicitation/create)."""
    result = await ctx.elicit("What is your name?", Answer)
    return f"{result.action}: {getattr(result, 'data', None)}"


async def name_ui(context, params):
    return types.ElicitResult(action="accept", content={"name": "Ada"})

async with Client(flights, mode="legacy", sampling_callback=host_llm) as old:
    print("legacy era", old.protocol_version, "->", (await old.call_tool(
        "find_best_flight", {"preferences": "cheap"})).content[0].text)

for mode in ["legacy", "auto"]:
    async with Client(legacy_demo, mode=mode, elicitation_callback=name_ui) as c:
        try:
            r = await c.call_tool("ask_name_old_style", {})
            print(f"{c.protocol_version}: ctx.elicit ->", r.content[0].text)
        except Exception as err:
            print(f"{c.protocol_version}: ctx.elicit -> {type(err).__name__}: {err}")

This is the migration lesson in code: **write new servers with resolvers or explicit `InputRequiredResult`s** (lesson 5.3), not with server-initiated calls.

### 7. Sampling with tools (also deprecated): needs `sampling.tools`
A server can give the host's model a set of tools scoped to the request. Without the `sampling.tools` capability, the SDK refuses to send such a request. With it, the callback receives `tools` and `toolChoice`, and the model can answer with `tool_use` blocks and `stopReason: "toolUse"`.

In [ ]:
weather = MCPServer("weather")
GET_WEATHER = types.Tool(name="get_weather", description="Get current weather for a city",
                         input_schema={"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]})


def plan(question: str) -> Sample:
    return Sample([types.SamplingMessage(role="user", content=types.TextContent(type="text", text=question))],
                  max_tokens=300, tools=[GET_WEATHER], tool_choice=types.ToolChoice(mode="auto"))


@weather.tool()
def compare_weather(question: str,
                    step: Annotated[types.CreateMessageResultWithTools, Resolve(plan)]) -> str:
    """Let the client's model decide which weather lookups to make."""
    return json.dumps(step.model_dump(by_alias=True, exclude_none=True, mode="json"))


async def tool_llm(context, params):
    print("  [host] tools offered:", [t.name for t in params.tools or []], "| toolChoice:", params.tool_choice.mode)
    return types.CreateMessageResultWithTools(
        role="assistant", model="claude-opus-5-5", stop_reason="toolUse",
        content=[types.ToolUseContent(type="tool_use", id="call_1", name="get_weather", input={"city": "Paris"}),
                 types.ToolUseContent(type="tool_use", id="call_2", name="get_weather", input={"city": "London"})])

async with Client(weather, sampling_callback=tool_llm) as c:          # sampling, but not sampling.tools
    try:
        await c.call_tool("compare_weather", {"question": "Paris or London?"})
    except MCPError as err:
        print("without sampling.tools ->", err.error.code, err.error.message)

tools_cap = types.SamplingCapability(tools=types.SamplingToolsCapability())
async with Client(weather, sampling_callback=tool_llm, sampling_capabilities=tools_cap) as c:
    out = await c.call_tool("compare_weather", {"question": "Paris or London?"})
    print("with sampling.tools    ->", out.content[0].text)

In a full loop the server would now run `get_weather` twice and send a *new* sampling request whose history ends with a user message of **only** `tool_result` blocks (`toolUseId` `call_1` and `call_2`), repeating until `endTurn`.

### 8. Roots (deprecated), and the recommended replacement
First the deprecated way. A `ListRoots()` resolver makes the server ask the client for its `file://` roots. Then the **migration**: the server takes its allowed directories from **configuration** at launch, and the tool takes the directory as a **parameter**. It validates every path by resolving it (which defeats `..` and symlinks) and checking it stays inside an allowed directory.

In [ ]:
import tempfile
from pathlib import Path

from mcp.server.mcpserver import ListRoots
from mcp.server.mcpserver.exceptions import ToolError

workspace = Path(tempfile.mkdtemp())
(workspace / "itineraries").mkdir()
(workspace / "itineraries" / "barcelona.md").write_text("Day 1: Sagrada Familia")
(workspace / "secrets.txt").write_text("do not leak")

# --- deprecated: roots via an input request ---
roots_server = MCPServer("roots-demo")


def fetch_roots() -> ListRoots:
    return ListRoots()


@roots_server.tool()
def list_workspace(roots: Annotated[types.ListRootsResult, Resolve(fetch_roots)]) -> list[str]:
    """List files under the client's roots (deprecated mechanism)."""
    files = []
    for root in roots.roots:
        base = Path(str(root.uri).removeprefix("file://"))
        files += [str(p.relative_to(base)) for p in sorted(base.rglob("*")) if p.is_file()]
    return files


async def my_roots(context):
    return types.ListRootsResult(roots=[types.Root(uri=f"file://{workspace / 'itineraries'}", name="Itineraries")])

async with Client(roots_server, list_roots_callback=my_roots) as c:
    print("roots-based listing:", (await c.call_tool("list_workspace", {})).structured_content)

# --- recommended: server configuration + explicit tool parameter ---
ALLOWED_DIRS = [(workspace / "itineraries").resolve()]       # e.g. from argv/env when the host launches the server
files_server = MCPServer("files")


@files_server.tool()
def read_text_file(directory: str, filename: str) -> str:
    """Read a text file from a directory the server is configured to allow."""
    target = (Path(directory) / filename).resolve()
    if not any(target.is_relative_to(allowed) for allowed in ALLOWED_DIRS):
        raise ToolError(f"{filename!r} resolves outside the allowed directories")
    return target.read_text()


async with Client(files_server) as c:
    ok = await c.call_tool("read_text_file", {"directory": str(workspace / "itineraries"), "filename": "barcelona.md"})
    print("allowed  :", ok.is_error, ok.content[0].text)
    bad = await c.call_tool("read_text_file", {"directory": str(workspace / "itineraries"), "filename": "../secrets.txt"})
    print("traversal:", bad.is_error, bad.content[0].text)

The traversal attempt fails *inside the server* and comes back as `isError: true` with a reason the model can read. We raise `ToolError` on purpose: `MCPServer` hides the message of any *unexpected* exception (a plain `ValueError` would reach the client only as "Error executing tool read_text_file"). Also note that the directory now appears in the **tool call** itself, so the host's approval UI shows the user exactly which folder the model wants to touch.

### 9. Optional: migrate sampling to a direct provider call
The replacement for sampling is ordinary code: the server calls an LLM API with *its own* key. When `ANTHROPIC_API_KEY` is set, this tool asks Claude directly. Otherwise it uses a deterministic fallback so the notebook still runs.

In [ ]:
import os

direct = MCPServer("flights-direct")


@direct.tool()
def recommend_flight(preferences: str) -> str:
    """Recommend a flight by calling the LLM provider directly (no sampling)."""
    prompt = f"Recommend the best flight in one sentence.\n{FLIGHTS}\nUser preferences: {preferences}"
    if os.environ.get("ANTHROPIC_API_KEY"):
        import anthropic
        response = anthropic.Anthropic().messages.create(
            model="claude-opus-5-5", max_tokens=16000, messages=[{"role": "user", "content": prompt}])
        return "".join(b.text for b in response.content if b.type == "text")
    return "(offline fallback) VY1307 — morning, one stop, cheapest that fits."


async with Client(direct) as c:   # note: no sampling_callback, no sampling capability needed
    print((await c.call_tool("recommend_flight", {"preferences": "morning, max 1 stop"})).content[0].text)

### Your turn
1. Add an `email: str` field with `format: "email"` to `BookingDetails` (use `pydantic.EmailStr` or `Field(json_schema_extra={"format": "email"})`) and check the rendered schema. Then try a nested model field and read the SDK's `TypeError`. Why does the spec forbid nesting?
2. Change `host_llm` so the "user" **rejects** the sampling request by returning `types.ErrorData(code=-1, message="User denied")`. What does `client.call_tool` raise, and what should a server do when sampling is refused?
3. In section 8, change `ToolError` to `ValueError` and compare the error text the client gets. Then add a symlink inside `itineraries` that points at `secrets.txt` and verify `resolve()` still blocks it.
4. Design question: the travel server needs the user's frequent-flyer **password** to apply miles. Sketch the input request you would send (mode, fields) and explain why form mode is not allowed.